In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
import regex as re
from transformers import BertTokenizer, AutoTokenizer
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

SEED = 42
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-lite-base-p2")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


In [2]:
df = pd.read_csv('../data/aes_dataset_5k_clean.csv')
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


## Split dataset

In [3]:
def split_dataset(df, train_ratio, valid_ratio, test_ratio):
    print("run split dataset...")
    subset_dataset = df['dataset_num'].unique()
    splits = {}
    for subset in subset_dataset:
        # get data by dataset_num
        subset_df = df[df['dataset_num'] == subset]

        # split dataset
        train_df, temp_df = train_test_split(subset_df, test_size=(1 - train_ratio), random_state=SEED, shuffle=True)
        valid_df, test_df = train_test_split(temp_df, test_size=test_ratio / (valid_ratio + test_ratio), random_state=SEED, shuffle=True)

        # save split dataset
        splits[subset] = {
            'train': train_df,
            'valid': valid_df,
            'test': test_df,
        }
    
    train_dataset = pd.concat([splits[subset]['train'] for subset in subset_dataset])
    valid_dataset = pd.concat([splits[subset]['valid'] for subset in subset_dataset])
    test_dataset = pd.concat([splits[subset]['test'] for subset in subset_dataset])

    return train_dataset, valid_dataset, test_dataset

train_dataset, valid_dataset, test_dataset = split_dataset(df, 0.8, 0.1, 0.1)
print("train dataset : ", train_dataset.shape)
print("valid dataset : ", valid_dataset.shape)
print("test dataset : ", test_dataset.shape)

run split dataset...
train dataset :  (1711, 6)
valid dataset :  (213, 6)
test dataset :  (238, 6)


## Dataset

In [30]:
class StudentAnswerDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.dataframe = dataframe
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataframe)
    
    def preprocess_text(self, text):
        # Remove extra whitespace
        text = ' '.join(text.split())
        # Convert to lowercase
        text = text.lower()
        # Remove special characters (keep punctuation)
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
        return text
    
    def __getitem__(self, index):
        student_answer = self.preprocess_text(str(self.dataframe.iloc[index]['answer']))
        score = self.dataframe.iloc[index]['normalized_score']

        encoding = self.tokenizer.encode_plus(
            student_answer,
            add_special_tokens=True,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors = 'pt'
        )

        encoding = {key: tensor.squeeze(0) for key, tensor in encoding.items()}
        encoding['labels'] = torch.tensor(score, dtype=torch.float)

        return encoding
    
    def get_max_length(self, index):
        student_answer = str(self.dataframe.iloc[index]['answer'])

        # concat input text
        encoding = self.tokenizer.encode_plus(
            student_answer,
            add_special_tokens=True,
            return_tensors = 'pt'
        )

        return encoding['input_ids'].flatten().shape[0]
    
dataset = StudentAnswerDataset(df, tokenizer)
dataset[0]

{'input_ids': tensor([    2,  1099,  1809, 29946, 21137,  1798, 29946,  2079,   737,  7725,
         29946,    41,   242,  6795,   825,     3,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

## Dataset 2

In [ ]:
class PairAnswerDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.dataframe = dataframe
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataframe)
    
    def preprocess_text(self, text):
        # Remove extra whitespace
        text = ' '.join(text.split())
        # Convert to lowercase
        text = text.lower()
        # Remove special characters (keep punctuation)
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
        return text
    
    def __getitem__(self, index):
        reference_answer = self.preprocess_text(str(self.dataframe.iloc[index]['reference_answer']))
        student_answer = self.preprocess_text(str(self.dataframe.iloc[index]['answer']))
        score = self.dataframe.iloc[index]['normalized_score']

        encoding = self.tokenizer.encode_plus(
            reference_answer,
            student_answer,
            add_special_tokens=True,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors = 'pt'
        )

        return encoding, torch.tensor(score, dtype=torch.float)
    
    def get_max_length(self, index):
        reference_answer = str(self.dataframe.iloc[index]['reference_answer'])
        student_answer = str(self.dataframe.iloc[index]['answer'])

        # concat input text
        encoding = self.tokenizer.encode_plus(
            reference_answer,
            student_answer,
            add_special_tokens=True,
            return_tensors = 'pt'
        )

        return encoding['input_ids'].flatten().shape[0]
    
# dataset = PairAnswerDataset(df, tokenizer)
# dataset[0]

## Create Dataset

In [31]:
# dataset = StudentAnswerDataset(df, tokenizer)
# df['max_length'] = df.index.map(dataset.get_max_length)
train_data = StudentAnswerDataset(train_dataset, tokenizer)
valid_data = StudentAnswerDataset(valid_dataset, tokenizer)
test_data = StudentAnswerDataset(test_dataset, tokenizer)

In [ ]:
def create_dataloader(train_data, valid_data, test_data):
    train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, generator=torch.Generator().manual_seed(SEED),)
    valid_dataloader = DataLoader(valid_data, batch_size=4, shuffle=False, generator=torch.Generator().manual_seed(SEED),)
    test_dataloader = DataLoader(test_data, batch_size=4, shuffle=False, generator=torch.Generator().manual_seed(SEED),)

    return train_dataloader, valid_dataloader, test_dataloader

train_dataloader, valid_dataloader, test_dataloader = create_dataloader(train_data, valid_data, test_data)

## Model 1.0

In [33]:
from transformers import AlbertForSequenceClassification

prebuild_model = AlbertForSequenceClassification.from_pretrained("indobenchmark/indobert-lite-base-p2", num_labels=1)
# prebuild_model

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-lite-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Model 1.1

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class RegressionModel(nn.Module):
    def __init__(self, model_name='bert-base-uncased'):
        super().__init__()
        # load pretrained model
        self.model = AutoModel.from_pretrained(model_name)
        # add regression layer
        self.dropout = nn.Dropout(p=0.1, inplace=False)
        self.regression_layer = nn.Linear(self.model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        score = self.regression_layer(cls_embedding)
        return score
    
customModel = RegressionModel('indobenchmark/indobert-lite-base-p2')
# customModel

## Pipeline 1

In [36]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
model = customModel.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)
criterion = torch.nn.MSELoss()
epochs = 1

In [44]:
for epoch in range(epochs):
    model.train()
    train_mse_loss = 0
    all_predictions = []
    all_targets = []

    for batchs in train_dataloader:
        optimizer.zero_grad()
        # move to device
        batchs = {k: v.to(device) for k, v in batchs.items()}

        # get prediction
        predictions = model(
            batchs['input_ids'], 
            batchs['attention_mask'], 
            batchs['token_type_ids']).squeeze(1)
        
        # calculate loss function
        loss = criterion(predictions, batchs['labels'])
        # backprop
        loss.backward()
        optimizer.step()

        # save training loss and metrik evaluation
        train_mse_loss += loss.item()
        all_predictions.extend(predictions.detach().cpu().numpy())
        all_targets.extend(batchs['labels'].detach().cpu().numpy())
    
    avg_train_loss = train_mse_loss / len(train_dataloader)
    mae = mean_absolute_error(all_targets, all_predictions)
    rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))
    pearson_corr, _ = pearsonr(all_targets, all_predictions)
    print(f"Epoch {epoch+1}/{epochs} - Avg training loss: {avg_train_loss:.4f}, MAE: {mae:.4}, RMSE: {rmse:.4}, Pearson Corr: {pearson_corr:.4}")

    # =============== EVAL PROCESS
    model.eval()
    total_mse_loss = 0
    all_predictions = []
    all_targets = []
    with torch.no_grad():
        for batchs in valid_dataloader:
            # move to device
            batchs = {k: v.to(device) for k, v in batchs.items()}

            # get prediction
            predictions = model(
                batchs['input_ids'], 
                batchs['attention_mask'], 
                batchs['token_type_ids']).squeeze(1)
            
            # calculate loss function
            loss = criterion(predictions, batchs['labels'])

            # save training loss and metrik evaluation
            total_mse_loss += loss.item()
            all_predictions.extend(predictions.detach().cpu().numpy())
            all_targets.extend(batchs['labels'].detach().cpu().numpy())

        avg_mse_loss = total_mse_loss / len(valid_dataloader)
        mae = mean_absolute_error(all_targets, all_predictions)
        rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))
        pearson_corr, _ = pearsonr(all_targets, all_predictions)
        print(f"Avg validation loss: {avg_mse_loss:.4f}, MAE: {mae:.4}, RMSE: {rmse:.4}, Pearson Corr: {pearson_corr:.4}")
        
        # update scheduler
        scheduler.step(avg_mse_loss)

Epoch 1/1 - Avg training loss: 0.0121, MAE: 0.08625, RMSE: 0.1102, Pearson Corr: 0.9144
Avg validation loss: 0.0163, MAE: 0.09564, RMSE: 0.1285, Pearson Corr: 0.865
